In [1]:
import geopandas as gpd
import pandas as pd
import re
from docx import Document
from rapidfuzz import fuzz, process

In [2]:
data_path = '../../../data/shapes'
shp_path = '../../../data/shapes/farm_boundaries.shp'
farmers_list = '../../../data/shapes/farmers_list.docx'
matched_names = '../../../data/shapes/matched_names_for_review.xlsx'

In [3]:
gdf = gpd.read_file(shp_path)

In [4]:
gdf.columns

Index(['name', 'melia_only', 'bounds_ok', 'geometry'], dtype='str')

In [5]:
gdf = gdf[['name', 'melia_only', 'bounds_ok', 'geometry']]

In [6]:
gdf['id'] = gdf.index + 1

In [7]:
no_name = gdf[gdf['name'].isna()]

In [8]:
doc = Document(farmers_list)

In [9]:
rows = []
for table in doc.tables:
    for row in table.rows[1:]:  # skip header
        cells = row.cells
        rows.append({
            "name": cells[1].text,
            "trees_given": cells[2].text,
            "trees_alive": cells[3].text
        })

doc_df = pd.DataFrame(rows)

In [10]:
doc_df.head()

,name,trees_given,trees_alive
0,Names,,
1,Rose Martha Mulaa,350,350
2,John Kithuku Mulu,150,104
3,Stella Kavindu Isaac,200,194
4,Priscah Martha Mutia,150,156


In [11]:
doc_df = doc_df.drop(index=0)

#### Name normalization

In [12]:
# Define unwanted substrings / tokens
UNWANTED_TERMS = [
    "awg",
    "farmer",
    "old farmer",
    "new",
    "ond",
    '-'
]

In [13]:
def normalize_name(name):
    if pd.isnull(name):
        return None
    name = name.lower()
    name = re.sub(r'[^\w\s]', '', name)  # remove punctuation
    for term in sorted(UNWANTED_TERMS, key=len, reverse=True):
        pattern = r'\b' + re.escape(term) + r'\b'
        name = re.sub(pattern, '', name)
    
    # Remove standalone 4-digit years
    name = re.sub(r'\b\d{4}\b', '', name)
    name = re.sub(r'\s+', ' ', name)     # collapse spaces
    return name.strip()

In [14]:
len(gdf)

155

In [15]:
# Remove null names
gdf = gdf[gdf["name"].notnull()]

# Split grouped names
gdf["name_split"] = gdf["name"].str.split(",")

# Explode into individual rows
gdf = gdf.explode("name_split")

# Normalize
gdf["name_clean"] = gdf["name_split"].apply(normalize_name)

# Remove empty
gdf = gdf[gdf["name_clean"].notnull()]

In [16]:
len(gdf)

160

In [17]:
gdf.columns

Index(['name', 'melia_only', 'bounds_ok', 'geometry', 'id', 'name_split',
       'name_clean'],
      dtype='str')

#### Clean docx names

In [18]:
len(doc_df)

174

In [19]:
doc_df["name_clean"] = doc_df["name"].apply(normalize_name)

In [20]:
doc_df[['name', 'name_clean']].iloc[:20]

,name,name_clean
1,Rose Martha Mulaa,rose martha mulaa
2,John Kithuku Mulu,john kithuku mulu
3,Stella Kavindu Isaac,stella kavindu isaac
4,Priscah Martha Mutia,priscah martha mutia
5,Daina Nditi Mutisya Ngambi,daina nditi mutisya ngambi
6,Justus Mutinda Mutia,justus mutinda mutia
7,Jeremiah Ngambi Mutisya,jeremiah ngambi mutisya
8,Maurice Mulaa Manzolo,maurice mulaa manzolo
9,Pius Mutia Mathuku,pius mutia mathuku
10,Lenah Koki Mutinda,lenah koki mutinda


In [21]:
len(doc_df)

174

#### Fuzzy matching

In [114]:
threshold = 70

shp_names = gdf["name_clean"].unique().tolist()
used_shp_names = set()

matches = []
unmatched_docx = []

for _, row in doc_df.iterrows():
    doc_name = row["name_clean"]

    if pd.isnull(doc_name):
        continue

    result = process.extractOne(
        doc_name,
        shp_names,
        scorer=fuzz.token_sort_ratio
    )

    if result:
        matched_name, score, _ = result

        if score >= threshold:
            matches.append({
                "docx_name": row["name"],
                "docx_clean": doc_name,
                "matched_shapefile_name": matched_name,
                "similarity_score": score
            })
            used_shp_names.add(matched_name)
        else:
            unmatched_docx.append(row["name"])
    else:
        unmatched_docx.append(row["name"])
        
matches_df = pd.DataFrame(matches)
unmatched_df = pd.DataFrame(unmatched_docx, columns=["name_not_in_shapefile"])

#### Output files

In [119]:
shapefile_only = list(set(shp_names) - used_shp_names)
len(shapefile_only)

shapefile_only_df = pd.DataFrame(
    shapefile_only,
    columns=["name_in_shapefile_not_in_docx"]
)
shapefile_only_df.head()
shapefile_only_df.to_excel("shapefile_not_in_docx.xlsx", index=False)

In [122]:
# A. Matched for manual review
matches_df.to_excel(f"{data_path}/matched_names_for_review.xlsx", index=False)

# B. Names in DOCX not in shapefile
unmatched_df.to_excel(f"{data_path}/docx_not_in_shapefile.xlsx", index=False)

# C. Normalized shapefile
gdf[['name_clean']].to_excel(f"{data_path}/shapefile_names.xlsx")

### Part 2

#### Merging names and trees given

In [22]:
gdf.head()

,name,melia_only,bounds_ok,geometry,id,name_split,name_clean
0,Naomie Mutuku,T,T,"POLYGON ((37.85681 -1.5947, 37.85699 -1.59471,...",1,Naomie Mutuku,naomie mutuku
1,Emma Mutie,T,T,"POLYGON ((37.85454 -1.59251, 37.85467 -1.59249...",2,Emma Mutie,emma mutie
2,Dorren Mutua,F,T,"POLYGON ((37.85088 -1.59412, 37.8509 -1.59402,...",3,Dorren Mutua,dorren mutua
3,Kinyao Musembi,F,T,"POLYGON ((37.86475 -1.58786, 37.86461 -1.58779...",4,Kinyao Musembi,kinyao musembi
4,Simon Mulu,F,T,"POLYGON ((37.87997 -1.59015, 37.88011 -1.59017...",5,Simon Mulu,simon mulu


In [23]:
gdf.drop(columns=['name_split'], inplace = True)

In [24]:
len(gdf)

160

In [25]:
matched_names = pd.read_excel(matched_names)

In [30]:
matched_names.head()

,docx_name,docx_clean,matched_shapefile_name
0,Rose Martha Mulaa,rose martha mulaa,rose mulaa
1,Stella Kavindu Isaac,stella kavindu isaac,stella isaac
2,Justus Mutinda Mutia,justus mutinda mutia,justus mutinda
3,Maurice Mulaa Manzolo,maurice mulaa manzolo,maurice manzolo
4,Pius Mutia Mathuku,pius mutia mathuku,pius mathuku


In [31]:
matched_names.drop(columns=['similarity_score'], inplace=True)

KeyError: "['similarity_score'] not found in axis"

In [33]:
match = gdf.merge(matched_names, how='left', left_on='name_clean', right_on='matched_shapefile_name')

In [34]:
len(match)

161

#### Add trees alive trees given

In [35]:
match.sample(10)

,name,melia_only,bounds_ok,geometry,id,name_clean,docx_name,docx_clean,matched_shapefile_name
121,James Mutie,T,T,"POLYGON ((37.86709 -1.60305, 37.86706 -1.60269...",117,james mutie,James Mutie,james mutie,james mutie
112,Joshua Mulwa,F,T,"POLYGON ((37.86862 -1.61479, 37.86891 -1.61479...",108,joshua mulwa,Joshua Mulwa Muyendei,joshua mulwa muyendei,joshua mulwa
87,Simon Sumaili - farmer,F,T,"POLYGON ((37.87405 -1.61845, 37.87427 -1.61851...",83,simon sumaili,Peter Simon Sumaili,peter simon sumaili,simon sumaili
92,Stella Isaac,T,T,"POLYGON ((37.86063 -1.61922, 37.86026 -1.61868...",88,stella isaac,Stella Kavindu Isaac,stella kavindu isaac,stella isaac
132,Mary Gibson,T,T,"POLYGON ((37.84545 -1.60166, 37.84519 -1.60174...",127,mary gibson,Mary Gibson,mary gibson,mary gibson
133,Dancun Mwengi,F,T,"POLYGON ((37.84492 -1.601, 37.84492 -1.60107, ...",128,dancun mwengi,Duncan Mwengi,duncan mwengi,dancun mwengi
30,Rose Ndinda Mutia,F,F,"POLYGON ((37.84401 -1.58508, 37.84389 -1.58506...",27,rose ndinda mutia,Rose Mutia,rose mutia,rose ndinda mutia
45,Daniel Kyule,F,T,"POLYGON ((37.87424 -1.56417, 37.87439 -1.56407...",42,daniel kyule,Daniel Musau Kyule,daniel musau kyule,daniel kyule
148,Benson Nyumu,F,T,"POLYGON ((37.87289 -1.62196, 37.87289 -1.62196...",143,benson nyumu,Benson Mutinda Nyumu,benson mutinda nyumu,benson nyumu
52,Paul Sila,T,T,"POLYGON ((37.85687 -1.55294, 37.857 -1.55259, ...",49,paul sila,Paul Sila Isika,paul sila isika,paul sila


In [36]:
doc_df.head()

,name,trees_given,trees_alive,name_clean
1,Rose Martha Mulaa,350,350,rose martha mulaa
2,John Kithuku Mulu,150,104,john kithuku mulu
3,Stella Kavindu Isaac,200,194,stella kavindu isaac
4,Priscah Martha Mutia,150,156,priscah martha mutia
5,Daina Nditi Mutisya Ngambi,300,564,daina nditi mutisya ngambi


In [37]:
doc_df.dtypes

name           str
trees_given    str
trees_alive    str
name_clean     str
dtype: object

In [38]:
doc_df['trees_given'] = pd.to_numeric(doc_df['trees_given'], errors='coerce')
doc_df['trees_alive'] = pd.to_numeric(doc_df['trees_alive'], errors='coerce')

In [39]:
doc_df = doc_df.astype({'trees_given' : 'Int64', 'trees_alive' : 'Int64'})

In [40]:
doc_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 174 entries, 1 to 174
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   name         174 non-null    str  
 1   trees_given  154 non-null    Int64
 2   trees_alive  151 non-null    Int64
 3   name_clean   174 non-null    str  
dtypes: Int64(2), str(2)
memory usage: 5.9 KB


#### Rejoin them

In [74]:
farmers_df = match.merge(doc_df, left_on='docx_clean', right_on='name_clean', how='left' )

In [75]:
farmers_df[farmers_df['name_clean_y'].isna()]

,name_x,melia_only,bounds_ok,geometry,id,name_clean_x,docx_name,docx_clean,matched_shapefile_name,name_y,trees_given,trees_alive,name_clean_y
10,Lucia Nyenze,T,T,"POLYGON ((37.87839 -1.57574, 37.87839 -1.57574...",11,lucia nyenze,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN
16,"Emily Nyamai, Maurice Manzolo, Rose Mulaa, Joe...",T,T,"POLYGON ((37.86484 -1.58302, 37.86482 -1.58301...",14,joel nyamai,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN
43,Titus Kakungu,F,F,"POLYGON ((37.87979 -1.57338, 37.87975 -1.57365...",40,titus kakungu,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN
48,Alex Kaluku Kituku,F,F,"POLYGON ((37.87355 -1.5612, 37.8748 -1.56107, ...",45,alex kaluku kituku,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN
50,Philip Musyoka,F,T,"POLYGON ((37.85553 -1.56401, 37.85577 -1.56379...",47,philip musyoka,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN
53,Carol Sila,T,T,"POLYGON ((37.85719 -1.55274, 37.85733 -1.55278...",50,carol sila,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN
70,Eunice Kamwene,T,T,"POLYGON ((37.87582 -1.64511, 37.87582 -1.64511...",66,eunice kamwene,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN
78,Gideon Mulei,F,T,"POLYGON ((37.86834 -1.62989, 37.86829 -1.62957...",74,gideon mulei,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN
96,Sameth Ngei,F,T,"POLYGON ((37.85472 -1.61851, 37.85487 -1.61849...",92,sameth ngei,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN
116,Veronica Musyoka,T,T,"POLYGON ((37.87067 -1.60674, 37.87084 -1.60678...",112,veronica musyoka,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN


In [76]:
len(farmers_df)

161

In [77]:
farmers_df.head()

,name_x,melia_only,bounds_ok,geometry,id,name_clean_x,docx_name,docx_clean,matched_shapefile_name,name_y,trees_given,trees_alive,name_clean_y
0,Naomie Mutuku,T,T,"POLYGON ((37.85681 -1.5947, 37.85699 -1.59471,...",1,naomie mutuku,Naomi Nicodemus Mutuku,naomi nicodemus mutuku,naomie mutuku,Naomi Nicodemus Mutuku,121,152,naomi nicodemus mutuku
1,Emma Mutie,T,T,"POLYGON ((37.85454 -1.59251, 37.85467 -1.59249...",2,emma mutie,Emma Mutie,emma mutie,emma mutie,Emma Mutie,150,145,emma mutie
2,Dorren Mutua,F,T,"POLYGON ((37.85088 -1.59412, 37.8509 -1.59402,...",3,dorren mutua,Dorrine Kasyoka Mutua,dorrine kasyoka mutua,dorren mutua,Dorrine Kasyoka Mutua,108,115,dorrine kasyoka mutua
3,Kinyao Musembi,F,T,"POLYGON ((37.86475 -1.58786, 37.86461 -1.58779...",4,kinyao musembi,Kinyao Musembi,kinyao musembi,kinyao musembi,Kinyao Musembi,150,33,kinyao musembi
4,Simon Mulu,F,T,"POLYGON ((37.87997 -1.59015, 37.88011 -1.59017...",5,simon mulu,Simon Mumo Mulu,simon mumo mulu,simon mulu,Simon Mumo Mulu,150,37,simon mumo mulu


In [78]:
farmers_df = farmers_df.drop(columns = ['name_y', 'matched_shapefile_name', 'name_x', 'docx_name', 'docx_clean'])

In [79]:
farmers_df.columns

Index(['melia_only', 'bounds_ok', 'geometry', 'id', 'name_clean_x',
       'trees_given', 'trees_alive', 'name_clean_y'],
      dtype='str')

In [80]:
farmers_df = farmers_df.rename(columns={'name_clean_y' : 'kamiti_cbo', 'name_clean_x' : 'awg_kml'})

In [81]:
farmers_df[farmers_df['kamiti_cbo'].isna()]

,melia_only,bounds_ok,geometry,id,awg_kml,trees_given,trees_alive,kamiti_cbo
10,T,T,"POLYGON ((37.87839 -1.57574, 37.87839 -1.57574...",11,lucia nyenze,<NA>,<NA>,NaN
16,T,T,"POLYGON ((37.86484 -1.58302, 37.86482 -1.58301...",14,joel nyamai,<NA>,<NA>,NaN
43,F,F,"POLYGON ((37.87979 -1.57338, 37.87975 -1.57365...",40,titus kakungu,<NA>,<NA>,NaN
48,F,F,"POLYGON ((37.87355 -1.5612, 37.8748 -1.56107, ...",45,alex kaluku kituku,<NA>,<NA>,NaN
50,F,T,"POLYGON ((37.85553 -1.56401, 37.85577 -1.56379...",47,philip musyoka,<NA>,<NA>,NaN
53,T,T,"POLYGON ((37.85719 -1.55274, 37.85733 -1.55278...",50,carol sila,<NA>,<NA>,NaN
70,T,T,"POLYGON ((37.87582 -1.64511, 37.87582 -1.64511...",66,eunice kamwene,<NA>,<NA>,NaN
78,F,T,"POLYGON ((37.86834 -1.62989, 37.86829 -1.62957...",74,gideon mulei,<NA>,<NA>,NaN
96,F,T,"POLYGON ((37.85472 -1.61851, 37.85487 -1.61849...",92,sameth ngei,<NA>,<NA>,NaN
116,T,T,"POLYGON ((37.87067 -1.60674, 37.87084 -1.60678...",112,veronica musyoka,<NA>,<NA>,NaN


In [82]:
farmers_df.sample(10)

,melia_only,bounds_ok,geometry,id,awg_kml,trees_given,trees_alive,kamiti_cbo
86,T,T,"POLYGON ((37.87501 -1.62255, 37.87517 -1.62225...",82,kanini ngala,150,95,rose kanini ngala
35,F,T,"POLYGON ((37.83835 -1.57967, 37.83867 -1.57972...",32,shadrack ngui mungami,200,89,shadrack ngui mungami
155,F,F,"POLYGON ((37.86913 -1.63026, 37.86931 -1.63032...",149,damaris mutheu james,150,113,damaris mutheu wambua
125,T,T,"POLYGON ((37.86348 -1.59682, 37.86356 -1.59645...",121,john katiwa nyamai,206,194,john katiwa nyamai
135,T,T,"POLYGON ((37.85615 -1.60612, 37.85629 -1.60601...",130,martha mule,150,98,martha mule
138,T,T,"POLYGON ((37.87963 -1.59828, 37.87981 -1.5983,...",133,safari mwamisi,150,24,abel safari mwamisi
94,F,F,"POLYGON ((37.86011 -1.61839, 37.86042 -1.61832...",90,isaac kiusya,200,174,isaac kitolo kiusya
124,T,T,"POLYGON ((37.86257 -1.59853, 37.86257 -1.59824...",120,beth titus,140,115,beth titus kithuku
17,F,T,"POLYGON ((37.86392 -1.58074, 37.86417 -1.58103...",15,samuel mativo,200,103,samuel mutisya mativo
123,T,T,"POLYGON ((37.86485 -1.59966, 37.86484 -1.59937...",119,florence mbuna,<NA>,<NA>,NaN


In [83]:
farmers_df.drop(columns = ['id'], inplace=True)

In [84]:
farmers_df.columns

Index(['melia_only', 'bounds_ok', 'geometry', 'awg_kml', 'trees_given',
       'trees_alive', 'kamiti_cbo'],
      dtype='str')

In [86]:
farmers_df = farmers_df[['kamiti_cbo', 'awg_kml', 'trees_given', 'trees_alive', 'melia_only', 'bounds_ok', 'geometry']]

In [87]:
farmers_df.head(
)

,kamiti_cbo,awg_kml,trees_given,trees_alive,melia_only,bounds_ok,geometry
0,naomi nicodemus mutuku,naomie mutuku,121,152,T,T,"POLYGON ((37.85681 -1.5947, 37.85699 -1.59471,..."
1,emma mutie,emma mutie,150,145,T,T,"POLYGON ((37.85454 -1.59251, 37.85467 -1.59249..."
2,dorrine kasyoka mutua,dorren mutua,108,115,F,T,"POLYGON ((37.85088 -1.59412, 37.8509 -1.59402,..."
3,kinyao musembi,kinyao musembi,150,33,F,T,"POLYGON ((37.86475 -1.58786, 37.86461 -1.58779..."
4,simon mumo mulu,simon mulu,150,37,F,T,"POLYGON ((37.87997 -1.59015, 37.88011 -1.59017..."


In [71]:
farmers_df.head()

,kamiti_cbo_name,awg_kml_name,trees_given,trees_alive,melia_only,bounds_ok,geometry
0,naomi nicodemus mutuku,naomie mutuku,121,152,T,T,"POLYGON ((37.85681 -1.5947, 37.85699 -1.59471,..."
1,emma mutie,emma mutie,150,145,T,T,"POLYGON ((37.85454 -1.59251, 37.85467 -1.59249..."
2,dorrine kasyoka mutua,dorren mutua,108,115,F,T,"POLYGON ((37.85088 -1.59412, 37.8509 -1.59402,..."
3,kinyao musembi,kinyao musembi,150,33,F,T,"POLYGON ((37.86475 -1.58786, 37.86461 -1.58779..."
4,simon mumo mulu,simon mulu,150,37,F,T,"POLYGON ((37.87997 -1.59015, 37.88011 -1.59017..."


In [89]:
farmers_df['awg_kml'] = farmers_df['awg_kml'].str.title()
farmers_df['kamiti_cbo'] = farmers_df['kamiti_cbo'].str.title()

In [90]:
farmers_df.head()

,kamiti_cbo,awg_kml,trees_given,trees_alive,melia_only,bounds_ok,geometry
0,Naomi Nicodemus Mutuku,Naomie Mutuku,121,152,T,T,"POLYGON ((37.85681 -1.5947, 37.85699 -1.59471,..."
1,Emma Mutie,Emma Mutie,150,145,T,T,"POLYGON ((37.85454 -1.59251, 37.85467 -1.59249..."
2,Dorrine Kasyoka Mutua,Dorren Mutua,108,115,F,T,"POLYGON ((37.85088 -1.59412, 37.8509 -1.59402,..."
3,Kinyao Musembi,Kinyao Musembi,150,33,F,T,"POLYGON ((37.86475 -1.58786, 37.86461 -1.58779..."
4,Simon Mumo Mulu,Simon Mulu,150,37,F,T,"POLYGON ((37.87997 -1.59015, 37.88011 -1.59017..."


In [91]:
farmers_df.to_file(f'{data_path}/awg_farmers_final.shp', index=False)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_11772\4129147925.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  farmers_df.to_file(f'{data_path}/awg_farmers_final.shp', index=False)
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'trees_given' to 'trees_give'
  ogr_write(
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'trees_alive' to 'trees_aliv'
  ogr_write(


In [93]:
farmers_df.to_csv(f"{data_path}/awg_farmers_final.csv", index=False)